# Boulder Orientation Annotation Tool

Shows each detected boulder (image crop + mask overlay) and lets you draw a line to mark the orientation you think is correct. Saves your annotations, then compares them against each model's measured orientation.

**How to annotate**: click two points to draw an orientation line across the boulder's long axis. The angle is computed from those two points. Press **n** to skip, **u** to undo the last click, **q** to stop.

**Setup**: point `PRED_SHP_DIR` at a folder of shapefile outputs (e.g. from `sam2_experiments.ipynb`), and `IN_RASTER` at the source raster so we can crop real image chips.

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import numpy as np
import matplotlib
matplotlib.use("TkAgg")   # interactive backend — use "Qt5Agg" if TkAgg not available
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.mask
from shapely.geometry import Polygon, box as shapely_box
from shapely import segmentize
from pathlib import Path
from tqdm import tqdm
import json
from datetime import datetime

try:
    from shptools_BOULDERING.geometry import fitEllipse
    from shptools_BOULDERING.geomorph import boulder_row
    HAS_SHPTOOLS = True
except ImportError:
    HAS_SHPTOOLS = False
    print("Warning: shptools not available — model orientations won't be computed")

print("Imports OK")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Source raster (the LRO NAC test raster used in the paper)
IN_RASTER = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")

# One shapefile per model — point each at your NMS-processed output
# These are the outputs from sam2_experiments.ipynb
work_dir = Path.home() / "tmp" / "YOLOv8BeyondEarth"
PRED_SHPS = {
    "YOLOv8":          sorted((work_dir / "exp_yolo_256").glob("*-downscaled-mask-nms.shp")),
    "SAM2 zero-shot":  sorted((work_dir / "exp_sam2_256").glob("*-downscaled-mask-nms.shp")),
    "SAM2 fine-tuned": sorted((work_dir / "exp_sam2_finetuned_256").glob("*-downscaled-mask-nms.shp")),
    "SAM2-auto":       sorted((work_dir / "exp_sam2_auto_256").glob("*-mask-nms.shp")),
}

OUT_DIR        = Path.home() / "tmp" / "boulder_annotations"
ANNOTATIONS_CSV= OUT_DIR / "user_annotations.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# How many boulders to annotate per session
N_TO_ANNOTATE = 50

# Crop pad around boulder bounding box (pixels in raster space)
PAD_PX = 30

# Only show boulders in this aspect-ratio range (same filter as paper)
AR_MIN, AR_MAX = 1.2, 2.0

for name, paths in PRED_SHPS.items():
    print(f"  {name}: {len(paths)} shapefiles found")
print(f"Raster: {'found' if IN_RASTER.exists() else 'NOT FOUND'}")

In [ ]:
# ── Load one model's predictions (used as the source for annotation boulders) ─
# We'll annotate boulders from SAM2 fine-tuned (best quality masks),
# but measure orientation from ALL models for the same boulder.

SOURCE_MODEL = "SAM2 fine-tuned"

gdfs = [gpd.read_file(p) for p in PRED_SHPS[SOURCE_MODEL]]
gdf_source = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
gdf_source["poly_area"] = gdf_source.geometry.area

print(f"Loaded {len(gdf_source)} {SOURCE_MODEL} detections")

# Load all model GDFs for cross-model orientation comparison later
model_gdfs = {}
for name, paths in PRED_SHPS.items():
    if not paths:
        continue
    gs = [gpd.read_file(p) for p in paths]
    model_gdfs[name] = gpd.GeoDataFrame(pd.concat(gs, ignore_index=True), crs=gs[0].crs)
    model_gdfs[name]["poly_area"] = model_gdfs[name].geometry.area
    print(f"  {name}: {len(model_gdfs[name])} detections")

In [ ]:
# ── Orientation pipeline (same as paper) ─────────────────────────────────────

def measure_orientation(geom, res=0.5):
    """fitEllipse → MRR → boulder_row. Returns (angle180, aspect_ratio) or None."""
    if not HAS_SHPTOOLS or geom is None or geom.is_empty:
        return None
    try:
        row_seg = pd.Series({"geometry": segmentize(geom, res)})
        ellipse_poly, _, _, _ = fitEllipse(row_seg)
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        _, _, long_ax, short_ax, _, _, _, angle180 = boulder_row(mrr_row)
        if short_ax < 1e-6:
            return None
        return float(angle180 % 180), float(long_ax / short_ax)
    except Exception:
        return None


def geom_iou(g1, g2):
    """Polygon IoU."""
    try:
        inter = g1.intersection(g2).area
        union = g1.union(g2).area
        return inter / union if union > 0 else 0.0
    except Exception:
        return 0.0


def find_matching_detection(target_geom, gdf, iou_thresh=0.3):
    """Find the detection in gdf with highest IoU vs target_geom."""
    best_iou, best_row = 0.0, None
    # Spatial index for speed
    candidates = gdf[gdf.geometry.intersects(target_geom.buffer(5))]
    for _, row in candidates.iterrows():
        iou = geom_iou(target_geom, row.geometry)
        if iou > best_iou:
            best_iou, best_row = iou, row
    return best_row if best_iou >= iou_thresh else None


print("Utility functions defined")

In [ ]:
# ── Pre-compute model orientations for all source boulders ────────────────────
# For each boulder in the source model, find matching detections in all other
# models and measure orientation. Saves to CSV so we don't re-run every session.

PRECOMPUTED_CSV = OUT_DIR / "model_orientations_precomputed.csv"

if PRECOMPUTED_CSV.exists():
    df_model_orient = pd.read_csv(PRECOMPUTED_CSV)
    print(f"Loaded precomputed orientations ({len(df_model_orient)} boulders)")
else:
    print("Pre-computing model orientations (this takes a few minutes)...")

    with rasterio.open(IN_RASTER) as src:
        res = src.res[0]

    rows = []
    for idx, src_row in tqdm(gdf_source.iterrows(), total=len(gdf_source), desc="Precomputing"):
        geom = src_row.geometry
        r_src = measure_orientation(geom, res)
        if r_src is None or not (AR_MIN <= r_src[1] <= AR_MAX):
            continue

        record = {
            "source_idx":      idx,
            "area":            src_row.poly_area,
            "geom_wkt":        geom.wkt,
            SOURCE_MODEL + "_angle": r_src[0],
            SOURCE_MODEL + "_ar":    r_src[1],
        }

        for model_name, mgdf in model_gdfs.items():
            if model_name == SOURCE_MODEL:
                continue
            match = find_matching_detection(geom, mgdf)
            if match is not None:
                r = measure_orientation(match.geometry, res)
                record[model_name + "_angle"] = r[0] if r else None
                record[model_name + "_ar"]    = r[1] if r else None
            else:
                record[model_name + "_angle"] = None
                record[model_name + "_ar"]    = None

        rows.append(record)

    df_model_orient = pd.DataFrame(rows)
    df_model_orient.to_csv(PRECOMPUTED_CSV, index=False)
    print(f"Saved {len(df_model_orient)} boulders to {PRECOMPUTED_CSV}")

print(f"Total annotatable boulders: {len(df_model_orient)}")

In [ ]:
# ── Interactive annotation session ────────────────────────────────────────────
# Shows each boulder crop. Click two points to define your orientation angle.
# Keyboard: n=skip, u=undo last click, q=quit

def crop_raster(raster_path, geom, pad_px=PAD_PX):
    """Crop raster to geometry bounds + pad. Returns (image_gray, transform, bounds_px)."""
    with rasterio.open(raster_path) as src:
        res = src.res[0]
        minx, miny, maxx, maxy = geom.bounds
        pad_world = pad_px * res
        crop_box = shapely_box(minx - pad_world, miny - pad_world,
                               maxx + pad_world, maxy + pad_world)
        try:
            out_img, out_transform = rasterio.mask.mask(src, [crop_box], crop=True)
        except Exception:
            return None, None, None
        raw = out_img[0].astype(np.float32)
        if raw.max() > 0:
            raw = (raw / raw.max() * 255).clip(0, 255).astype(np.uint8)
        else:
            raw = raw.astype(np.uint8)
        return raw, out_transform, crop_box.bounds


def geom_to_px(geom, crop_bounds, image_shape):
    """Convert georeferenced geometry to pixel coordinates in the cropped image."""
    minx_c, miny_c, maxx_c, maxy_c = crop_bounds
    h, w = image_shape
    scale_x = w / (maxx_c - minx_c)
    scale_y = h / (maxy_c - miny_c)
    pts = np.array(geom.exterior.coords[:-1])
    px = (pts[:, 0] - minx_c) * scale_x
    py = h - (pts[:, 1] - miny_c) * scale_y   # flip y (raster origin top-left)
    return np.stack([px, py], axis=-1).astype(np.int32)


def angle_from_two_points(p1, p2):
    """Angle of the line from p1 to p2 in [0, 180)."""
    dx, dy = p2[0] - p1[0], p2[1] - p1[1]
    angle = np.degrees(np.arctan2(-dy, dx)) % 180   # -dy because image y is flipped
    return float(angle)


# Load or init annotations file
if ANNOTATIONS_CSV.exists():
    df_annot = pd.read_csv(ANNOTATIONS_CSV)
    already_done = set(df_annot["source_idx"].tolist())
    print(f"Resuming — {len(df_annot)} annotations already saved")
else:
    df_annot = pd.DataFrame()
    already_done = set()
    print("Starting fresh annotation session")

# Sample boulders to annotate (skip already done)
candidates = df_model_orient[
    ~df_model_orient["source_idx"].isin(already_done)
].sample(frac=1, random_state=42).head(N_TO_ANNOTATE)
print(f"Will show {len(candidates)} boulders this session")

In [ ]:
# ── Run the annotation loop ───────────────────────────────────────────────────

new_annotations = []

for _, boulder in candidates.iterrows():
    from shapely import wkt as shapely_wkt
    geom = shapely_wkt.loads(boulder["geom_wkt"])

    crop_img, crop_tf, crop_bounds = crop_raster(IN_RASTER, geom)
    if crop_img is None:
        continue

    pts_px = geom_to_px(geom, crop_bounds, crop_img.shape)

    # Show model angles in title
    angle_strs = []
    for model_name in model_gdfs.keys():
        a = boulder.get(model_name + "_angle")
        angle_strs.append(f"{model_name}: {a:.1f}°" if pd.notna(a) else f"{model_name}: –")

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(crop_img, cmap="gray", vmin=0, vmax=255)

    # Draw the source model's mask outline in cyan
    ax.plot(np.append(pts_px[:, 0], pts_px[0, 0]),
            np.append(pts_px[:, 1], pts_px[0, 1]),
            color="cyan", lw=1.5, alpha=0.8)

    # Show each model's measured orientation as a line through the centroid
    cy_px, cx_px = crop_img.shape[0] // 2, crop_img.shape[1] // 2
    LINE_COLORS = {"YOLOv8": "steelblue", "SAM2 zero-shot": "tomato",
                   "SAM2 fine-tuned": "orange", "SAM2-auto": "mediumpurple"}
    half_len = crop_img.shape[0] * 0.35
    for model_name, mcolor in LINE_COLORS.items():
        a = boulder.get(model_name + "_angle")
        if pd.notna(a):
            rad = np.radians(a)
            dx, dy = np.cos(rad) * half_len, -np.sin(rad) * half_len
            ax.plot([cx_px - dx, cx_px + dx], [cy_px - dy, cy_px + dy],
                    color=mcolor, lw=1.5, alpha=0.7, label=f"{model_name}: {a:.0f}°")

    ax.legend(loc="upper right", fontsize=7, framealpha=0.7)
    ax.set_title(
        f"Boulder #{int(boulder['source_idx'])}  —  Click two points along the LONG AXIS\n"
        "(n=skip, u=undo last click, q=quit session)",
        fontsize=9
    )
    ax.axis("off")

    clicked = []
    skipped = [False]
    quit_flag = [False]

    def on_click(event):
        if event.inaxes != ax or event.xdata is None:
            return
        clicked.append((event.xdata, event.ydata))
        ax.plot(event.xdata, event.ydata, "y+", ms=12, mew=2)
        if len(clicked) == 2:
            p1, p2 = clicked
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], "y-", lw=2)
            plt.draw()

    def on_key(event):
        if event.key == "n":
            skipped[0] = True
            plt.close()
        elif event.key == "u" and clicked:
            clicked.pop()
            ax.lines[-1].remove()
            plt.draw()
        elif event.key == "q":
            quit_flag[0] = True
            plt.close()

    fig.canvas.mpl_connect("button_press_event", on_click)
    fig.canvas.mpl_connect("key_press_event", on_key)
    plt.tight_layout()
    plt.show(block=True)

    if quit_flag[0]:
        print("Session ended by user")
        break

    if skipped[0] or len(clicked) < 2:
        continue

    user_angle = angle_from_two_points(clicked[0], clicked[1])
    rec = {
        "source_idx":      int(boulder["source_idx"]),
        "user_angle":      user_angle,
        "annotated_at":    datetime.now().isoformat(),
    }
    for model_name in list(model_gdfs.keys()) + [SOURCE_MODEL]:
        rec[model_name + "_angle"] = boulder.get(model_name + "_angle")
    new_annotations.append(rec)
    print(f"  #{int(boulder['source_idx'])}: user={user_angle:.1f}°  "
          + "  ".join(f"{m}: {boulder.get(m+'_angle', None):.1f}°"
                      if pd.notna(boulder.get(m+"_angle")) else f"{m}: –"
                      for m in model_gdfs.keys()))

# Save
if new_annotations:
    df_new = pd.DataFrame(new_annotations)
    df_annot = pd.concat([df_annot, df_new], ignore_index=True)
    df_annot.to_csv(ANNOTATIONS_CSV, index=False)
    print(f"\nSaved {len(new_annotations)} new annotations → {ANNOTATIONS_CSV}")
    print(f"Total annotations: {len(df_annot)}")
else:
    print("No new annotations")

In [ ]:
# ── Comparison: user angles vs. each model ─────────────────────────────────────
# Run this after annotating enough boulders (aim for 30+).

if not ANNOTATIONS_CSV.exists():
    print("No annotations yet — run the annotation loop first")
else:
    df_annot = pd.read_csv(ANNOTATIONS_CSV)
    print(f"Loaded {len(df_annot)} annotations")

    model_cols = [c.replace("_angle", "") for c in df_annot.columns
                  if c.endswith("_angle") and c != "user_angle"]

    n_plots = len(model_cols) + 1
    fig, axes = plt.subplots(2, n_plots, figsize=(4 * n_plots, 8))

    BINS = np.linspace(0, 180, 37)
    CX   = (BINS[:-1] + BINS[1:]) / 2
    COLORS = {"user": "gold", "YOLOv8": "steelblue", "SAM2 zero-shot": "tomato",
              "SAM2 fine-tuned": "darkorange", "SAM2-auto": "mediumpurple"}

    # Row 0: orientation histograms
    ax_user = axes[0, 0]
    user_angles = df_annot["user_angle"].dropna().values
    counts, _ = np.histogram(user_angles, bins=BINS)
    ax_user.bar(CX, counts, width=4.5, color="gold", edgecolor="white")
    ax_user.axhline(len(user_angles)/len(CX), color="k", ls="--", lw=0.8)
    ax_user.set(xlim=(0,180), xticks=[0,45,90,135,180], xlabel="Orientation (°)", ylabel="Count")
    ax_user.set_title(f"Your annotations\n(n={len(user_angles)})", fontsize=9)
    ax_user.spines[["top","right"]].set_visible(False)

    for col_i, model_name in enumerate(model_cols, start=1):
        col = model_name + "_angle"
        ax = axes[0, col_i]
        angles = df_annot[col].dropna().values
        color  = COLORS.get(model_name, "gray")
        counts, _ = np.histogram(angles, bins=BINS)
        ax.bar(CX, counts, width=4.5, color=color, edgecolor="white")
        ax.axhline(len(angles)/len(CX), color="k", ls="--", lw=0.8)
        ax.set(xlim=(0,180), xticks=[0,45,90,135,180], xlabel="Orientation (°)", ylabel="Count")
        ax.set_title(f"{model_name}\n(n={len(angles)})", fontsize=9)
        ax.spines[["top","right"]].set_visible(False)

    # Row 1: scatter (user angle vs. model angle) per boulder
    axes[1, 0].axis("off")
    axes[1, 0].text(0.5, 0.5, "Your angles\n(histogram above)",
                    ha="center", va="center", transform=axes[1,0].transAxes, fontsize=10)

    for col_i, model_name in enumerate(model_cols, start=1):
        col = model_name + "_angle"
        ax = axes[1, col_i]
        valid = df_annot[["user_angle", col]].dropna()
        color = COLORS.get(model_name, "gray")
        if not valid.empty:
            ax.scatter(valid["user_angle"], valid[col], s=20, alpha=0.6, color=color)
            ax.plot([0, 180], [0, 180], "k--", lw=0.8)
            errors = np.minimum(
                np.abs(valid["user_angle"].values - valid[col].values),
                180 - np.abs(valid["user_angle"].values - valid[col].values)
            )
            mae = np.mean(errors)
            ax.text(0.05, 0.95, f"MAE vs you: {mae:.1f}°",
                    transform=ax.transAxes, fontsize=9, va="top")
        ax.set(xlim=(0,180), ylim=(0,180),
               xticks=[0,45,90,135,180], yticks=[0,45,90,135,180],
               xlabel="Your angle (°)", ylabel=f"{model_name} angle (°)")
        ax.set_title(f"You vs. {model_name}", fontsize=9)
        ax.spines[["top","right"]].set_visible(False)

    plt.suptitle("Your orientation annotations vs. each model's measurement", fontsize=12)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "user_vs_models.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved user_vs_models.png")

In [ ]:
# ── Error breakdown: which boulders do you and the model disagree on most? ────

if ANNOTATIONS_CSV.exists() and len(df_annot) > 0:
    # Add per-boulder error columns
    for model_name in model_cols:
        col = model_name + "_angle"
        err_col = model_name + "_err"
        diff = np.abs(df_annot["user_angle"] - df_annot[col])
        df_annot[err_col] = np.minimum(diff, 180 - diff)

    # Show the top disagreements
    err_cols = [m + "_err" for m in model_cols if m + "_err" in df_annot.columns]
    if err_cols:
        df_annot["max_model_err"] = df_annot[err_cols].max(axis=1)
        top_disagreements = df_annot.nlargest(10, "max_model_err")[["source_idx", "user_angle"] + err_cols]
        print("Top 10 boulders where models and you disagree most:")
        print(top_disagreements.to_string(index=False))

    # Summary stats per model
    print("\nMean absolute error vs. your annotations:")
    for model_name in model_cols:
        err_col = model_name + "_err"
        if err_col in df_annot.columns:
            mae = df_annot[err_col].dropna().mean()
            print(f"  {model_name:25s}: {mae:.1f}°")